In [1]:
query = "I just found out about the course. can I still join?"

In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/home/jobin/exploration/Agentic AI/llm-zoomcamp-2026/02-vector-search/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
q1 = "Can I still join the course after the start date?"
q1_vec = model.encode(q1)

In [9]:
q2 = "what should be my course of medication to get well soon?"
q2_vec = model.encode(q2)

In [10]:
q1_vec.dot(q2_vec)

np.float32(0.13924251)

In [11]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [13]:
q1_vec.dot(dv)

np.float32(0.32332397)

In [14]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)


In [15]:
v2.dot(dv)

np.float32(0.019730434)

### embeddd the FAQ dataset ###

In [16]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-22 06:26:40--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-22 06:26:40 (88.7 MB/s) - ‘ingest.py’ saved [888/888]



In [17]:
from ingest import load_faq_data

In [18]:
documents = load_faq_data()

In [20]:
from tqdm.auto import tqdm

In [21]:
texts = []

for doc in tqdm(documents):
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

  0%|          | 0/1350 [00:00<?, ?it/s]

In [26]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [29]:
import numpy as np

In [30]:
X = np.array(vectors)

In [31]:
X.shape

(1350, 384)

In [36]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [44]:
scores = X.dot(v_query)

In [46]:
idx = np.argmax(scores)

In [47]:
print(f"idx={idx},score={scores[idx]}")

idx=2,score=0.7629410028457642


In [54]:
top5 = np.argsort(scores)
top5 = top5[-5:]
top5 = top5[::-1] ##reverse

In [55]:
top5

array([  2, 625, 907, 538,   7])

In [58]:
documents[top5[0]]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [59]:
for idx in top5:
    print(f"index={idx},score={scores[idx]}")
    print(documents[idx])

index=2,score=0.7629410028457642
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}
index=625,score=0.7579370737075806
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}
index=907,score=0.7192131876945496
{'course': 'mac

In [62]:
np.argsort(-scores)[:5]

array([  2, 625, 907, 538,   7])

In [63]:
top5

array([  2, 625, 907, 538,   7])